# 🌑 LunarSight — Notebook 06: Full Pipeline

Run the complete LangGraph orchestrated pipeline:
Agent 1 → Agent 2 → Agent 3 → Agent 4 → Coverage Check → Agent 5

With automatic retry logic for coverage and path failures.

---

In [ ]:
# === Setup ===
import os, torch
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/Lunar-Sight'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/YOUR_USERNAME/Lunar-Sight.git {REPO_DIR}
os.chdir(os.path.join(REPO_DIR, 'Lunar-Sight'))
!pip install -q -r requirements_colab.txt

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# === Run Full Pipeline ===
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-30s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S'
)

from orchestrator import build_graph

graph = build_graph()

initial_state = {
    'mission_config_path': 'config/mission_config.yaml',
    'current_agent': 'agent1',
}

print('Starting full pipeline...')
result = graph.invoke(initial_state)

print('\n' + '=' * 60)
print('PIPELINE COMPLETE')
print('=' * 60)
for key in sorted(result.keys()):
    if 'status' in key or 'path' in key.lower():
        print(f'  {key}: {result[key]}')

In [ ]:
# === Final Visualization ===
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Slope
if result.get('slope_path') and os.path.exists(result['slope_path']):
    slope = np.load(result['slope_path'])
    axes[0, 0].imshow(slope, cmap='RdYlGn_r', vmin=0, vmax=20)
    axes[0, 0].set_title('Terrain Slope')

# 2. CPR
if result.get('cpr_l_path') and os.path.exists(result['cpr_l_path']):
    cpr = np.load(result['cpr_l_path'])
    axes[0, 1].imshow(cpr, cmap='RdYlBu_r', vmin=0, vmax=3)
    axes[0, 1].set_title('L-band CPR')

# 3. Ice Mask
if result.get('ice_mask_path') and os.path.exists(result['ice_mask_path']):
    ice = np.load(result['ice_mask_path'])
    axes[0, 2].imshow(ice, cmap='Blues')
    axes[0, 2].set_title(f'Ice Mask ({np.sum(ice)} pixels)')

# 4. Confidence
if result.get('confidence_map_path') and os.path.exists(result['confidence_map_path']):
    conf = np.load(result['confidence_map_path'])
    axes[1, 0].imshow(conf, cmap='viridis', vmin=0, vmax=1)
    axes[1, 0].set_title('Confidence Map')

# 5. Path on slope
path = result.get('traverse_path', [])
if path and result.get('slope_path'):
    slope = np.load(result['slope_path'])
    axes[1, 1].imshow(slope, cmap='terrain')
    px = [w['x'] for w in path]
    py = [w['y'] for w in path]
    axes[1, 1].plot(px, py, 'r-', linewidth=2)
    axes[1, 1].plot(px[0], py[0], 'g^', ms=12)
    axes[1, 1].plot(px[-1], py[-1], 'r*', ms=15)
    axes[1, 1].set_title(f'Traverse ({len(path)} waypoints)')

# 6. Battery
if path:
    bat = [w['battery_wh'] for w in path]
    axes[1, 2].plot(bat, 'g-', lw=2)
    axes[1, 2].fill_between(range(len(bat)), bat, alpha=0.3, color='green')
    axes[1, 2].set_title('Battery Profile')
    axes[1, 2].set_ylabel('Wh')

plt.suptitle('LunarSight — Full Pipeline Results', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()